In [ ]:
!pip install catboost

In [ ]:
import seaborn as sns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path,'Q3_data.csv')
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
for i in df.columns:
  df[i]=df[i].fillna(df[i].mean())

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=['object']).columns
categorical_cols

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler
numerical_cols = df.select_dtypes(include=['number']).columns.drop('Target')
print('data before scaling:\n', numerical_cols) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
df[numerical_cols] = standard_scaler.fit_transform(df[numerical_cols]) # Apply fit_transform

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()
check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', axis=1)
y = df['Target']

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
results = {'acc': [], 'f1': []}

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  print(f"Training CatBoost Classifier...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  results['acc'].append(accuracy)
  results['f1'].append(f1)

In [ ]:
print(f"  Accuracy:  {np.mean(results['acc']):.4f}")
print(f"  F1-Score:  {np.mean(results['f1']):.4f}")
print(f"However, since the data is imbalanced. The best metric here would be F1-Score which is {np.mean(results['f1']):.2f}")

In [ ]:
# Task 1: Write your code here:
catboost_model = model
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(48, 44))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:
for i in sorted_catboost_importance:
  print(i[0])

In [ ]:
print(f"The most important feature is {sorted_catboost_importance[0][0]}")

In [ ]:
r = {'acc': [], 'f1': []}

In [ ]:
# Task Bonus: Write your code here:
X = df[['P_2']]


n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model_golden = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  print(f"Training CatBoost Classifier...")
  model_golden.fit(X_train, y_train) # train
  y_pred = model_golden.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  r['acc'].append(accuracy)
  r['f1'].append(f1)


In [ ]:
print(f"  Accuracy:  {np.mean(r['acc']):.4f}")
print(f"  F1-Score:  {np.mean(r['f1']):.4f}")